# Оцінювання моделей

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from data.loading import load_baskets
from evaluation.metrics import evaluate, split_baskets
from models.catalog import build_catalog
from models.popularity import Popularity
from models.recommend import recommend, register, save_models

In [2]:
K = 10

## Дані, каталог, поділ кошиків

In [3]:
train = load_baskets("train")
val_splits = split_baskets(load_baskets("val"))
test_splits = split_baskets(load_baskets("test"))

catalog = build_catalog(train)
print(f"каталог: {len(catalog):,} товарів; кошиків для оцінки: val {len(val_splits):,}, test {len(test_splits):,}")

каталог: 14,603 товарів; кошиків для оцінки: val 7,677, test 18,771


## Бейзлайн: популярність

In [4]:
register("popularity", Popularity(train, catalog))
display(recommend([], "popularity", K))

,position,product_id,product_name,aisle,score
0,1,24852,Banana,fresh fruits,10596
1,2,13176,Bag of Organic Bananas,fresh fruits,8413
2,3,21137,Organic Strawberries,fresh fruits,5890
3,4,21903,Organic Baby Spinach,packaged vegetables fruits,5357
4,5,47209,Organic Hass Avocado,fresh fruits,4800
5,6,47766,Organic Avocado,fresh fruits,3861
6,7,47626,Large Lemon,fresh fruits,3492
7,8,16797,Strawberries,fresh fruits,3199
8,9,26209,Limes,fresh fruits,3161
9,10,27845,Organic Whole Milk,milk,3031


In [5]:
results = {"popularity": evaluate("popularity", test_splits, K)}
pd.DataFrame(results).T.round(4)

,recall@10,ndcg@10,n_baskets
popularity,0.0637,0.0563,18771.0


## Спільні покупки (co-occurrence)
Одна матриця «скільки разів товари купили разом» і три способи нормалізації:
- **conditional** — P(j | i): частка кошиків з i, де є й j;
- **cosine** — C[i, j] / √(n_i · n_j);
- **lift** — у скільки разів частіше разом, ніж випадково.

Підбираємо `min_support` та `ordered` для кожної нормалізації на **val**.

In [6]:
from models.cooccurrence import CoOccurrence
from models.recommend import register

val_results = {"popularity": evaluate("popularity", val_splits, K)}
params = {} 
for similarity in ["conditional", "cosine", "lift"]:
    for min_support in [1, 3, 5, 10, 30, 100]:
        for ordered in [True, False]:
            name = f"{similarity}_min{min_support}" + ("_ordered" if ordered else "")
            params[name] = (similarity, min_support, ordered)
            register("candidate", CoOccurrence(train, catalog, similarity, min_support, ordered))
            val_results[name] = evaluate("candidate", val_splits, K)

val_table = pd.DataFrame(val_results).T
val_table.round(4)

,recall@10,ndcg@10,n_baskets
popularity,0.0640,0.0563,7677.0
conditional_min1_ordered,0.0814,0.0719,7677.0
conditional_min1,0.0869,0.0740,7677.0
conditional_min3_ordered,0.0842,0.0739,7677.0
conditional_min3,0.0868,0.0741,7677.0
conditional_min5_ordered,0.0837,0.0734,7677.0
conditional_min5,0.0865,0.0738,7677.0
conditional_min10_ordered,0.0812,0.0724,7677.0
conditional_min10,0.0840,0.0722,7677.0
conditional_min30_ordered,0.0751,0.0681,7677.0


In [7]:
# Найкращий min_support для кожного варіанту (нормалізація + ordered), за NDCG на val
val_table["variant"] = val_table.index.str.replace(r"_min\d+", "", regex=True)
best = val_table.drop("popularity").groupby("variant")["ndcg@10"].idxmax()
best

variant
conditional                    conditional_min3
conditional_ordered    conditional_min3_ordered
cosine                              cosine_min5
cosine_ordered             cosine_min10_ordered
lift                                lift_min100
lift_ordered                lift_min100_ordered
Name: ndcg@10, dtype: str

In [8]:
for name in best:
    register(name, CoOccurrence(train, catalog, *params[name]))
    results[name] = evaluate(name, test_splits, K)

pd.DataFrame(results).T.round(4)

,recall@10,ndcg@10,n_baskets
popularity,0.0637,0.0563,18771.0
conditional_min3,0.0841,0.0734,18771.0
conditional_min3_ordered,0.0817,0.0718,18771.0
cosine_min5,0.0835,0.0733,18771.0
cosine_min10_ordered,0.0730,0.0632,18771.0
lift_min100,0.0613,0.0539,18771.0
lift_min100_ordered,0.0585,0.0517,18771.0


## EASE (лінійна регресія)
Кожен товар передбачаємо через усі інші товари кошика, з ridge-регуляризацією λ.
Стаття: Steck, *Embarrassingly Shallow Autoencoders for Sparse Data*, 2019 (https://arxiv.org/abs/1905.03375).

Підбираємо λ на **val**. Кожне навчання — обернення матриці 14.6k × 14.6k, ~1–2 хв.

In [9]:
from models.ease import EASE
from models.recommend import register

ease_val = {}
for lam in [500, 1000, 2000, 5000]:
    register("candidate", None)
    register("candidate", EASE(train, catalog, lam))
    ease_val[f"ease_lam{lam}"] = evaluate("candidate", val_splits, K)

ease_val = pd.DataFrame(ease_val).T
ease_val.round(4)

,recall@10,ndcg@10,n_baskets
ease_lam500,0.0878,0.0774,7677.0
ease_lam1000,0.0875,0.0777,7677.0
ease_lam2000,0.0878,0.0772,7677.0
ease_lam5000,0.0880,0.0773,7677.0


## Фінальне порівняння на test

In [10]:
best_lam = int(ease_val["ndcg@10"].idxmax().removeprefix("ease_lam"))
register("candidate", None) 

register(f"ease_lam{best_lam}", EASE(train, catalog, best_lam))
results[f"ease_lam{best_lam}"] = evaluate(f"ease_lam{best_lam}", test_splits, K)

pd.DataFrame(results).T.round(4)

,recall@10,ndcg@10,n_baskets
popularity,0.0637,0.0563,18771.0
conditional_min3,0.0841,0.0734,18771.0
conditional_min3_ordered,0.0817,0.0718,18771.0
cosine_min5,0.0835,0.0733,18771.0
cosine_min10_ordered,0.0730,0.0632,18771.0
lift_min100,0.0613,0.0539,18771.0
lift_min100_ordered,0.0585,0.0517,18771.0
ease_lam1000,0.0852,0.0759,18771.0


## Збереження результатів для дашборду
Таблиця метрик + кількість тестових кошиків + покриття (частка прихованих товарів test, що є в каталозі).

In [11]:
import json

hidden_items = test_splits["hidden"].explode()

quality = {
    "n_test_baskets": len(test_splits),
    "catalog_size": len(catalog),
    "coverage": round(hidden_items.isin(catalog).mean(), 4),
    "metrics": pd.DataFrame(results).T.drop(columns="n_baskets").round(4).to_dict(orient="index"),
}

(ROOT / "artifacts").mkdir(exist_ok=True)
with open(ROOT / "artifacts" / "quality.json", "w") as f:
    json.dump(quality, f, indent=2)
quality

{'n_test_baskets': 18771,
 'catalog_size': 14603,
 'coverage': np.float64(0.9336),
 'metrics': {'popularity': {'recall@10': 0.0637, 'ndcg@10': 0.0563},
  'conditional_min3': {'recall@10': 0.0841, 'ndcg@10': 0.0734},
  'conditional_min3_ordered': {'recall@10': 0.0817, 'ndcg@10': 0.0718},
  'cosine_min5': {'recall@10': 0.0835, 'ndcg@10': 0.0733},
  'cosine_min10_ordered': {'recall@10': 0.073, 'ndcg@10': 0.0632},
  'lift_min100': {'recall@10': 0.0613, 'ndcg@10': 0.0539},
  'lift_min100_ordered': {'recall@10': 0.0585, 'ndcg@10': 0.0517},
  'ease_lam1000': {'recall@10': 0.0852, 'ndcg@10': 0.0759}}}

## Збереження моделей для дашборду
Популярність + найкращі варіанти conditional, cosine і EASE (усі навчені на train).

In [12]:
save_models({
    "popularity": "popularity",
    "conditional": best["conditional"],
    "cosine": best["cosine"],
    "ease": f"ease_lam{best_lam}",
})